# Phase 0 — interactive pilot and measurement companion

This notebook is an **interactive audit of the Phase 0 pilot**, not the authoritative
source of final estimates. The scientific protocol and engineering provenance are in
`docs/phase0-design.md` and `docs/phase0-implementation.md`; the notebook reads cached
artefacts from `results/p0_pilot/metrics/` and makes no model or network calls.

**Design.** The sample comprises the 43 judged TREC DL 2019 queries. All 45 unordered
pairs in each query's BM25 top 10 yield 1,900 observed pairs (some queries return fewer
than ten candidates). Each pair is presented in both orders. An order-consistent A/B
choice is a *decisive verdict*; disagreement becomes a tie. Ten lexical axioms are
evaluated on the same canonical pairs. Qwen 3.6 35B-A3B is the primary judge and
Flan-T5-large is an architectural contrast, not an independent replication sample.

The pilot asks three measurement questions:

1. How often does each axiom actually *apply* to a real BM25 top-10 pair?
2. When an axiom does apply, how often does it agree with the LLM judge?
3. How robust are pairwise verdicts to presentation order, and how often do cycles
   occur among fully decisive triangles?

> **Superseded artefact handling.** The historical cache contains invalid PROX1 and
> PROX2 cells from before the axiom-stage correction. This notebook excludes both axioms
> from every active table and figure. The correction chronology belongs in
> `docs/research-logbook.md`; corrected Phase 1 artefacts support later proximity claims.

All estimates here are descriptive and exploratory: 43 queries are the sampling units,
pairs within a query are dependent, and this pilot does not report query-bootstrap
intervals or multiplicity-adjusted tests.

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

from axiomrank.paths import results_dir

INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
COLORS = {"models/qwen3.6-35B-A3B-AWQ": "#2a78d6", "google/flan-t5-large": "#1baf7a"}
LABELS = {"models/qwen3.6-35B-A3B-AWQ": "Qwen 3.6 35B", "google/flan-t5-large": "Flan-T5-large"}
PRIMARY = "models/qwen3.6-35B-A3B-AWQ"
SUPERSEDED_AXIOMS = {"PROX1", "PROX2"}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": INK2,
    "axes.edgecolor": BASE, "axes.grid": False, "font.size": 11,
})

metrics_root = results_dir("p0_pilot") / "metrics"
if not metrics_root.exists():
    raise FileNotFoundError(f"Missing cached Phase 0 metrics: {metrics_root}")
agreement, consistency = {}, {}
for d in sorted(metrics_root.iterdir()):
    model = d.name.replace("__", "/")
    frame = pd.read_csv(d / "agreement.csv").set_index("axiom")
    agreement[model] = frame.drop(index=SUPERSEDED_AXIOMS, errors="ignore")
    consistency[model] = json.loads((d / "consistency.json").read_text())

missing = set(COLORS) - set(agreement)
if missing:
    raise FileNotFoundError(f"Missing model artefacts: {sorted(missing)}")

agr = pd.concat(agreement, names=["model"])
agr.loc[PRIMARY, ["coverage", "n_evaluable", "agreement"]].round(3)

## 1. Axiom applicability (coverage)

For axiom $j$, coverage is $N^{-1}\sum_i 1[a_{ij} \neq 0]$: the proportion of all
canonical pairs for which its preconditions hold and it expresses a non-neutral
preference. Coverage depends on the sampled retrieval pool and axiom implementation,
not on the LLM judge. It is not an effectiveness measure.

In [ ]:
cov = agreement[PRIMARY]["coverage"].sort_values()

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.barh(cov.index, cov * 100, height=0.58, color="#2a78d6")
for y, v in enumerate(cov * 100):
    ax.text(v + 1.2, y, f"{v:.1f}%" if v >= 0.5 else f"{v:.2f}%",
            va="center", color=INK2, fontsize=10)
ax.set_xlim(0, 100)
ax.set_xlabel("pairs where the axiom expresses a preference (%)")
ax.set_title("Axiom coverage on BM25 top-10 pairs (DL19, 1,900 pairs)",
             loc="left", fontsize=12, color=INK)
ax.spines[["top", "right", "bottom"]].set_visible(False)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(length=0)
plt.tight_layout()
plt.show()

In this top-10 pool, TFC1 and several valid proximity axioms have substantial coverage,
whereas TFC3 (2 pairs) and M-TDC (about 12 evaluable pairs) have too little support
for stable agreement estimates. This is evidence about **strict preconditions under
this sampling regime**, not grounds to discard those axioms generally. It motivates
the relaxed-precondition analysis in Phase 1. Superseded PROX1/PROX2 cells are not
displayed.

## 2. Agreement with the LLM judge, where the axiom applies

For axiom $j$, conditional agreement is the fraction of pairs with $a_{ij}\neq0$ and
a decisive LLM verdict for which the signs match. The estimand therefore differs across
axioms because each fires on a different subset. The 0.5 line is the balanced binary
reference, not necessarily the empirical majority baseline. Grey marks $n<30$, an
exploratory display rule rather than a statistical reliability threshold.

In [ ]:
MIN_N = 30
order = (agreement[PRIMARY]
         .assign(reliable=lambda d: d.n_evaluable >= MIN_N)
         .sort_values(["reliable", "agreement"])
         .index.tolist())

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.axvline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)))
ax.text(0.502, len(order) - 0.25, "chance", color=MUTED, fontsize=9, va="top")

for model, color in COLORS.items():
    df = agreement[model].loc[order]
    reliable = df.n_evaluable >= MIN_N
    for rel, c in [(reliable, color), (~reliable, GRID)]:
        sub = df[rel]
        ax.scatter(sub.agreement, [order.index(a) for a in sub.index],
                   s=72, color=c, edgecolors=SURFACE, linewidths=1.5, zorder=3,
                   label=LABELS[model] if rel is reliable else None)

ylabels = [f"{a}  (n={agreement[PRIMARY].loc[a, 'n_evaluable']:,})" for a in order]
ax.set_yticks(range(len(order)), ylabels)
ax.set_xlim(0.25, 0.95)
ax.set_xlabel("agreement with LLM pairwise verdict")
ax.set_title("Axiom–judge agreement (grey = too few evaluable pairs)",
             loc="left", fontsize=12, color=INK)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

Three descriptive observations motivate Phase 1:

- Among valid, sufficiently covered estimates, PROX3 has the largest observed
  conditional agreement (~0.68–0.71). No uncertainty interval or paired comparison is
  available here, so this is not a claim that it is statistically best.
- **TFC1 sits at or below chance** (~0.47) despite being the highest-coverage axiom.
  This supports only a conditional top-10 finding; BM25 range restriction is a plausible
  explanation to test with the Phase 1 wide-gap control.
- PROX1 and PROX2 are excluded rather than algebraically repaired; the corrected Phase 1
  run is the active source for those axioms.

Cross-model similarity is useful robustness evidence, but both models see the same
queries and documents and use different prompts/backends; it is not independent
replication and does not identify architecture as the cause of any difference.

## 3. Presentation robustness and conditional cyclicity

Position consistency is the proportion of unordered pairs whose A/B choice identifies
the same document after swapping presentation order. Cyclicity is computed only among
triangles for which all three collapsed pair verdicts are decisive. Because this
conditioning removes inconsistent edges, a low cycle rate does **not** establish global
transitivity of the raw judge.

In [ ]:
rows = {}
for model, s in consistency.items():
    rows[LABELS[model]] = {
        "position consistency": f"{s['position_consistency']:.1%}",
        "complete triangles (of sampled)": f"{s['triangle_survival']:.1%}",
        "non-transitive triangles": f"{s['nontransitivity_rate']:.2%}",
        "mean latency / pair": f"{s['mean_latency_ms']:,.0f} ms",
    }
pd.DataFrame(rows)[["Qwen 3.6 35B", "Flan-T5-large"]]

Order-swap robustness is only 67–71%, so collecting both orders and collapsing
disagreements to ties remains necessary. Among the selected complete-decisive
triangles, fewer than 0.4% are cyclic. Qwen is descriptively more position-consistent
and faster in this environment, but latency is hardware/backend-specific and the cycle
counts are too small for a strong comparative claim.

### Takeaways

1. Strict axioms have sharply different coverage in BM25 top-10 pairs; low-coverage
   estimates require relaxation or substantially more data.
2. The valid pilot estimates suggest heterogeneous conditional agreement and justify a
   collection-replicated, query-bootstrap analysis rather than a single aggregate score.
3. Moderate position sensitivity makes order swap part of the measurement definition.
   Conditional cycle rarity supports aggregation operationally but is not proof of
   transitivity.
4. Qwen is selected as the Phase 1 primary on the documented sanity, consistency, and
   operational criteria; Flan-T5-large remains a contrast model. This is a study-design
   choice, not evidence of general model superiority.
5. Superseded proximity cells remain outside the active analysis; their chronology is
   documented in the research logbook.